# SChain Neon v2 — safe deep-discovery resume to Tier 5

This notebook continues **the existing `bmw-supply-chain-v2` experiment**. It never creates a replacement. Take a Neon backup and stop every worker before enabling the one-time migration cell.

In [ ]:
EXPERIMENT_ID = 'bmw-supply-chain-v2'
CONFIG_PATH = 'config/distributed_neon_deep_discovery.yaml'
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
SAVE_TO_NEO4J = False
TASKS_THIS_SESSION = 40
AUTO_CONTINUE = True
COLAB_RUNTIME_MINUTES = 110
APPLY_ONE_TIME_MIGRATION = False  # set True only after backup + worker shutdown
assert EXPERIMENT_ID == 'bmw-supply-chain-v2' and not SAVE_TO_NEO4J

In [ ]:
import os, pathlib, subprocess
REPO_DIR = pathlib.Path('/content/Schain')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'mirza', '--single-branch', 'https://github.com/ErfanBabakhani/Schain.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'diff', '--quiet'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'mirza'], check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[postgres]'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'vllm'], check=True)

In [ ]:
from google.colab import userdata
value = userdata.get('NEON_DATABASE_URL')
if not value: raise RuntimeError('Missing NEON_DATABASE_URL Colab Secret')
os.environ['NEON_DATABASE_URL'] = value
print('Neon credential loaded without displaying it.')

## Read-only status and extension plan
These commands do not migrate the schema or change experiment rows. Inspect the plan and confirm that Toyota, every Tier-1 company, and the terminal Tier-2 companies are present.

In [ ]:
subprocess.run(['python','-m','schain.distributed.maintenance','status','--experiment-id',EXPERIMENT_ID,'--json'], check=True)
subprocess.run(['python','-m','schain.distributed.maintenance','extend','--experiment-id',EXPERIMENT_ID,'--new-max-tier','5','--topup-tier','1','--plan'], check=True)

## One-time confirmed migration
Run only after a Neon backup and after all status output shows zero active workers and leases. Re-running is idempotent.

In [ ]:
if APPLY_ONE_TIME_MIGRATION:
    subprocess.run(['python','-m','schain.distributed.maintenance','extend','--experiment-id',EXPERIMENT_ID,'--new-max-tier','5','--topup-tier','1','--apply','--confirm-id',EXPERIMENT_ID], check=True)
else:
    print('Migration not applied. Set APPLY_ONE_TIME_MIGRATION=True only after backup and worker shutdown.')

## Start Qwen, then safely continue worker sessions
Use the same Qwen endpoint as the original experiment. The worker configuration has `resume_existing_only=True`, uses the persisted deep-discovery profile, and cannot create a missing experiment.

In [ ]:
import json, logging, time, httpx, uuid
VLLM_URL = 'http://127.0.0.1:8000/v1'
def vllm_ready():
    try: return httpx.get(VLLM_URL + '/models', timeout=2).status_code == 200
    except Exception: return False
if not vllm_ready():
    log_path = pathlib.Path('/content') / f'vllm-{uuid.uuid4().hex[:8]}.log'
    log_handle = log_path.open('w')
    process = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--host','127.0.0.1','--port','8000','--max-model-len','8192','--gpu-memory-utilization','0.88'], stdout=log_handle, stderr=subprocess.STDOUT)
    deadline = time.time() + 600
    while time.time() < deadline and not vllm_ready():
        if process.poll() is not None: raise RuntimeError(f'vLLM exited; inspect {log_path}')
        time.sleep(5)
    if not vllm_ready(): raise TimeoutError(f'vLLM not ready; inspect {log_path}')
print('Original Qwen endpoint ready.')
from schain.config import load_config
from schain.distributed.profile import validate_deep_discovery
from schain.distributed.runner import run_distributed_expand
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s', force=True)
config = load_config(CONFIG_PATH)
config['state']['experiment_id'] = EXPERIMENT_ID
config['llm']['model'] = MODEL
config['llm']['base_url'] = VLLM_URL
config['save']['enabled'] = SAVE_TO_NEO4J
assert config['distributed']['resume_existing_only'] is True
print(json.dumps(validate_deep_discovery(config), indent=2))
deadline = time.monotonic() + COLAB_RUNTIME_MINUTES * 60
while True:
    result = run_distributed_expand(config, tasks_this_session=TASKS_THIS_SESSION)
    print(json.dumps(result, indent=2, default=str), flush=True)
    status = result['experiment']['status']
    if status in {'completed', 'failed'} or not AUTO_CONTINUE or time.monotonic() >= deadline:
        break
    print('Session checkpointed; continuing the same experiment...', flush=True)

In [ ]:
subprocess.run(['python','-m','schain.distributed.maintenance','status','--experiment-id',EXPERIMENT_ID,'--json'], check=True)